# 02. Router / Orchestrator Agent (`ResearchRouter`) Demo

### Hierarchical Supervisor Architecture
The **ResearchRouter** is an `OrchestratorAgent` that wraps the specialist agents (`EarningsAnalyst`, `MarketAnalyst`, `NewsAnalyst`) as callable delegation tools (`delegate_to_earningsanalyst`, `delegate_to_marketanalyst`, `delegate_to_newsanalyst`).

- **Dynamic ReAct Delegation**: Rather than relying on rigid keyword dictionaries, the orchestrator uses the real LLM to inspect the user's inquiry and autonomously decide which specialist(s) to call.
- **Targeted Inquiries**: Calls only the requested specialist (e.g. only EarningsAnalyst for fundamental queries).
- **Multi-Domain Inquiries**: Sequentially or concurrently invokes multiple specialists and synthesizes their findings.
- **Extensibility**: Supports adding new specialist sub-agents dynamically via `add_subagent()`.

In [1]:
import sys
from pathlib import Path

# Resolve src directory
project_root = Path.cwd().parent if Path.cwd().name == 'demo' else Path.cwd()
sys.path.insert(0, str(project_root / 'src'))

from research_agent.agents.router import router_agent
from research_agent.interfaces import get_llm

# Initialize OrchestratorAgent with real LLM and specialist subagents
llm = get_llm()
orchestrator = router_agent(llm=llm)
print(f"Orchestrator Agent initialized: {orchestrator.name}")
print(f"Registered Sub-agents: {list(orchestrator.subagents.keys())}")
print(f"Delegation Tools: {list(orchestrator.tools.keys())}")

Orchestrator Agent initialized: ResearchRouter
Registered Sub-agents: ['EarningsAnalyst', 'MarketAnalyst', 'NewsAnalyst']
Delegation Tools: ['delegate_to_earningsanalyst', 'delegate_to_marketanalyst', 'delegate_to_newsanalyst']


## Scenario 1: Targeted Domain Delegation (Earnings / Fundamentals)
When the user asks specifically about earnings, the orchestrator invokes `delegate_to_earningsanalyst`.

In [2]:
# Scenario 1: Targeted Domain Delegation (Earnings / Fundamentals)
q1 = "What were NVDA's latest quarterly revenue, gross margin, and EPS?"
print(f"Sending Query: '{q1}'\n")

resp_1 = orchestrator.run(q1)
print("--- ORCHESTRATOR SYNTHESIS ---")
print(resp_1.content)

print("\n--- DELEGATION TRACE ---")
for step in resp_1.trace:
    if step.step_type == "tool_call":
        print(f"DELEGATION ACTION: {step.content.get('tool')}({step.content.get('arguments')})")
    elif step.step_type == "final_answer":
        print(f"FINAL ANSWER: Generated ({len(str(step.content))} characters)")

Sending Query: 'What were NVDA's latest quarterly revenue, gross margin, and EPS?'

--- ORCHESTRATOR SYNTHESIS ---
**NVIDIA Corp. (NVDA) – Latest Quarter (Q3 FY 2026, ended July 31 2026)**  

| Metric | Reported Figure |
|--------|------------------|
| **Revenue** | **$96.22 B** |
| **Gross Margin** | **74.98 %** (gross profit $72.14 B) |
| **Diluted EPS** | **$2.22** per share |

*Data source:* Directly retrieved from the `get_earnings_data` tool for NVDA’s most recent quarter (ended 31 Jul 2026). No other calculations or estimates were applied.

--- DELEGATION TRACE ---
DELEGATION ACTION: delegate_to_earningsanalyst({'task_description': "Retrieve NVIDIA Corporation (NVDA) latest quarterly earnings data, specifically the most recent quarter's revenue, gross margin, and earnings per share (EPS). Use get_earnings_data and, if needed, search_sec_filings for the latest 10-Q filing. Return the exact figures as provided by the tools, with citations."})
FINAL ANSWER: Generated (437 character

## Scenario 2: Multi-Domain Delegation (Technicals & News Catalysts)
When the user asks about price trend and recent news, the orchestrator delegates to `MarketAnalyst` and `NewsAnalyst`.

In [3]:
# Scenario 2: Multi-Domain Delegation (Technicals & News Catalysts)
q2 = "Examine NVDA price trend, moving averages, and recent news catalysts."
print(f"Sending Query: '{q2}'\n")

resp_2 = orchestrator.run(q2)
print("--- ORCHESTRATOR SYNTHESIS ---")
print(resp_2.content)

print("\n--- DELEGATION TRACE ---")
for step in resp_2.trace:
    if step.step_type == "tool_call":
        print(f"DELEGATION ACTION: {step.content.get('tool')}({step.content.get('arguments')})")
    elif step.step_type == "final_answer":
        print(f"FINAL ANSWER: Generated ({len(str(step.content))} characters)")

Sending Query: 'Examine NVDA price trend, moving averages, and recent news catalysts.'

--- ORCHESTRATOR SYNTHESIS ---
**NVDA – Integrated Market‑Technical & News‑Catalyst Brief (as of 12 Oct 2026)**  

---

## 1. Price Trend & Technical Overview  

| Metric | Value | Interpretation |
|--------|-------|----------------|
| **Current price** | $229.28 | Trading near the top of its 52‑week range. |
| **52‑wk high / low** | $243.37 / $164.27 | ~5.8 % below the high; ~39.6 % above the low. |
| **50‑day SMA** | $221.94 | Price **$7.34 above** the short‑term average → short‑term bullish bias. |
| **200‑day SMA** | $201.83 | Price **$27.45 above** the long‑term average → confirmed medium‑term uptrend. |
| **Trend label (tool)** | **Bullish Uptrend** | Aligns with price above both SMAs. |
| **14‑day RSI** | 53.03 | In the neutral band (30‑70); no over‑bought or over‑sold signal. |
| **Beta (vs. S&P 500)** | 2.216 | ~2.2× market volatility – higher upside potential but also higher downside risk.

## Scenario 3: Dynamic Specialist Subagent Registration
Demonstrates how new sub-agents can be dynamically added at runtime using `orchestrator.add_subagent(...)`.

In [4]:
# Scenario 3: Dynamic Specialist Subagent Registration
from research_agent.agent_framework import Agent

# Dynamically register a new Macro Analyst agent
macro_sub = Agent(
    name="MacroAnalyst",
    role="Analyze macroeconomic factors such as Fed interest rates, Treasury yields, and inflation.",
    llm=llm,
)
orchestrator.add_subagent(macro_sub)
print(f"Updated Sub-agents: {list(orchestrator.subagents.keys())}")
print(f"Updated Delegation Tools: {list(orchestrator.tools.keys())}")

Updated Sub-agents: ['EarningsAnalyst', 'MarketAnalyst', 'NewsAnalyst', 'MacroAnalyst']
Updated Delegation Tools: ['delegate_to_earningsanalyst', 'delegate_to_marketanalyst', 'delegate_to_newsanalyst', 'delegate_to_macroanalyst']
